# MLflow — śledzenie eksperymentów ML

**Problem bez MLflow:** trenujesz model 20 razy, zmieniasz hiperparametry, i po tygodniu nie pamiętasz który run dał AUC 0.87 i jakich parametrów użyłeś.

**MLflow rozwiązuje:** automatyczne logowanie parametrów, metryk, modeli i artefaktów — z UI do porównywania runów.

UI dostępne na: **http://localhost:5001**

## 1. Połączenie z serwerem MLflow

In [1]:
import os
import mlflow

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000"))

EXPERIMENT = "mlflow_demo"
mlflow.set_experiment(EXPERIMENT)

print(f"URI:        {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT}")
print(f"UI:         http://localhost:5001")

2026/04/21 15:40:01 INFO mlflow.tracking.fluent: Experiment with name 'mlflow_demo' does not exist. Creating a new experiment.


URI:        http://mlflow:5000
Experiment: mlflow_demo
UI:         http://localhost:5001


## 2. Dataset

Używamy Iris — prostego, żeby skupić się na MLflow, nie na modelu.

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Zbiór treningowy: {X_train.shape}")
print(f"Zbiór testowy:    {X_test.shape}")
print(f"Klasy: {set(y)}")

Zbiór treningowy: (120, 4)
Zbiór testowy:    (30, 4)
Klasy: {0, 1, 2}


## 3. Ręczny run - pełna kontrola

Najbardziej explicite podejście: sam decydujesz co logujesz.
Dobre żeby zrozumieć co MLflow przechowuje.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

params = {
    "solver": "lbfgs",
    "max_iter": 200,
    "C": 1.0,
    "random_state": 42,
}

with mlflow.start_run(run_name="logistic_regression_bazowy"):

    # 1. Parametry — hiperparametry modelu
    mlflow.log_params(params)

    # 2. Tag — dowolna metadana, przydatna do filtrowania w UI
    mlflow.set_tag("autor", "szkolenie")
    mlflow.set_tag("dataset", "iris")

    # 3. Trenowanie
    model = LogisticRegression(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # 4. Metryki
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average="weighted")
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_weighted", f1)

    # 5. Model jako artefakt — artifact_path wymagany w MLflow 2.x
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        input_example=X_train[:3],
    )

    run_id = mlflow.active_run().info.run_id

print(f"Run ID:   {run_id}")
print(f"Accuracy: {acc:.3f}")
print(f"F1:       {f1:.3f}")
print(f"UI:       http://localhost:5001/#/experiments/...")

2026/04/21 15:40:08 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Run ID:   277ff31ccb794e939fa528b249133044
Accuracy: 1.000
F1:       1.000
UI:       http://localhost:5001/#/experiments/...


## 4. Logowanie metryk iteracyjnie

Przydatne gdy trenujesz sieć neuronową i chcesz śledzić loss po każdej epoce.

In [4]:
with mlflow.start_run(run_name="symulacja_epochow"):
    mlflow.log_param("lr", 0.01)

    # Symulacja — w praktyce to byłby training loop sieci
    import numpy as np
    for epoch in range(10):
        fake_loss = 1.0 * np.exp(-0.3 * epoch) + np.random.normal(0, 0.02)
        fake_acc  = 1 - fake_loss * 0.5
        mlflow.log_metric("train_loss", fake_loss, step=epoch)
        mlflow.log_metric("train_acc",  fake_acc,  step=epoch)

    print("Run zapisany — w UI zobaczysz wykresy loss/acc po epokach")

Run zapisany — w UI zobaczysz wykresy loss/acc po epokach


## 5. Artefakty - logowanie dowolnych plików

Oprócz modelu możesz zapisać: wykres, CSV z predykcjami, raport.

In [7]:
import tempfile, os, json
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

with mlflow.start_run(run_name="z_artefaktami"):
    model2 = LogisticRegression(max_iter=200).fit(X_train, y_train)
    y_pred2 = model2.predict(X_test)

    mlflow.sklearn.log_model(model2, artifact_path="model", input_example=X_train[:3])
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred2))

    # Artefakt 1: macierz pomyłek jako HTML (Plotly)
    cm = confusion_matrix(y_test, y_pred2)
    fig = go.Figure(go.Heatmap(z=cm, colorscale="Blues",
                               text=cm, texttemplate="%{text}"))
    fig.update_layout(title="Confusion Matrix", width=400, height=400)

    with tempfile.TemporaryDirectory() as tmp:
        html_path = os.path.join(tmp, "confusion_matrix.html")
        fig.write_html(html_path)
        mlflow.log_artifact(html_path, artifact_path="plots")

        # Artefakt 2: params jako JSON (czytelne archiwum)
        meta = {"params": params, "n_train": len(X_train), "n_test": len(X_test)}
        meta_path = os.path.join(tmp, "run_meta.json")
        with open(meta_path, "w") as f:
            json.dump(meta, f, indent=2)
        mlflow.log_artifact(meta_path, artifact_path="meta")

    print("Artefakty zapisane — sprawdź w UI zakładkę Artifacts")

/usr/local/lib/python3.10/site-packages/_distutils_hack/__init__.py:15: UserWarning:

Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.

/usr/local/lib/python3.10/site-packages/_distutils_hack/__init__.py:30: UserWarning:

Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml



Artefakty zapisane — sprawdź w UI zakładkę Artifacts


## 6. Autolog - jedna linijka zamiast wszystkiego

`mlflow.sklearn.autolog()` automatycznie loguje: parametry, metryki, model, feature importance.
Wada: mniej kontroli nad tym co i jak się zapisuje.

In [8]:
from sklearn.ensemble import RandomForestClassifier

# log_datasets=False — unikamy konfliktu setuptools/distutils w Python 3.10
mlflow.sklearn.autolog(log_models=True, log_datasets=False, silent=True)


with mlflow.start_run(run_name="random_forest_autolog"):
    rf = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42)
    rf.fit(X_train, y_train)

    auto_run_id = mlflow.active_run().info.run_id

mlflow.sklearn.autolog(disable=True)

print(f"Run ID: {auto_run_id}")
print("Sprawdź w UI — params + metrics + model zapisane bez żadnego mlflow.log_*")

AssertionError: /usr/local/lib/python3.10/distutils/core.py

## 7. Porównanie runów programatycznie

UI pozwala porównać runy klikaniem. Możesz też zrobić to z kodu.

In [9]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT],
    order_by=["metrics.accuracy DESC"],
)

cols = ["run_id", "tags.mlflow.runName", "metrics.accuracy", "metrics.f1_weighted",
        "params.max_iter", "params.C"]
available = [c for c in cols if c in runs.columns]
print(runs[available].to_string(index=False))

                          run_id        tags.mlflow.runName  metrics.accuracy  metrics.f1_weighted params.max_iter params.C
d6aa2c85e7604f59a95b4286ad47fd7f              z_artefaktami               1.0                  NaN            None     None
408baa7b643d40d3b07ae181beaf417b              z_artefaktami               1.0                  NaN            None     None
277ff31ccb794e939fa528b249133044 logistic_regression_bazowy               1.0                  1.0             200      1.0
44e8aa98eead4daf845978f7ad7fbc01          symulacja_epochow               NaN                  NaN            None     None


## 8. Ładowanie modelu z MLflow

Kiedy model jest już w produkcji - ładujesz go po `run_id` lub nazwie z rejestru.

In [11]:
# Załaduj model z pierwszego runu (przez run_id)
model_uri = f"runs:/{run_id}/model"
loaded_model = mlflow.sklearn.load_model(model_uri)

y_pred_loaded = loaded_model.predict(X_test)
print(f"Predykcje z załadowanego modelu: {y_pred_loaded}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_loaded):.3f}")
print()
print("→ Ten sam model, inny proces — tak działa deployment")

2026/04/21 15:45:58 INFO mlflow.store.artifact.artifact_repo: The progress bar can be disabled by setting the environment variable MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR to false


Predykcje z załadowanego modelu: [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]
Accuracy: 1.000

→ Ten sam model, inny proces — tak działa deployment


## 9. Model Registry - wersjonowanie modeli

Registry to katalog modeli z wersjami i **aliasami**.

Zamiast dzielić się `run_id` - dzielisz się nazwą i aliasem: `models:/iris_demo@champion`.

Serwis produkcyjny zawsze ładuje alias `champion` -  żeby wdrożyć nową wersję,
wystarczy przestawić alias w Registry. Zero zmian w kodzie serwisu.

In [12]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
MODEL_NAME = "iris_demo"

# Zarejestruj model z istniejącego runu
result = mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name=MODEL_NAME,
)
print(f"Zarejestrowano: {MODEL_NAME} v{result.version}")

Successfully registered model 'iris_demo'.
2026/04/21 15:46:25 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris_demo, version 1


Zarejestrowano: iris_demo v1


Created version '1' of model 'iris_demo'.


In [13]:
result

<ModelVersion: aliases=[], creation_timestamp=1776786385544, current_stage='None', description='', last_updated_timestamp=1776786385544, name='iris_demo', run_id='277ff31ccb794e939fa528b249133044', run_link='', source='mlflow-artifacts:/1/277ff31ccb794e939fa528b249133044/artifacts/model', status='READY', status_message='', tags={}, user_id='', version='1'>

In [14]:
# Przypisz alias — zamiast deprecated stage (Staging/Production)
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="champion",
    version=result.version,
)
print(f"Alias 'champion' → {MODEL_NAME} v{result.version}")

# Sprawdź wersje i aliasy
for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
    aliases = v.aliases if hasattr(v, "aliases") else []
    print(f"  v{v.version}  aliases={aliases}  run_id={v.run_id[:8]}...")

Alias 'champion' → iris_demo v1
  v1  aliases=[]  run_id=277ff31c...


In [15]:
# Załaduj model po aliasie — tak działa deployment
champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")

y_pred = champion.predict(X_test)

print(f"Accuracy (@champion): {accuracy_score(y_test, y_pred):.3f}")
print()
print("Żeby 'wdrożyć' nową wersję: set_registered_model_alias(..., alias='champion', version=2)")
print("Serwis produkcyjny ładuje zawsze '@champion' — nie wie nic o numerach wersji")

2026/04/21 15:47:32 INFO mlflow.store.artifact.artifact_repo: The progress bar can be disabled by setting the environment variable MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR to false


Accuracy (@champion): 1.000

Żeby 'wdrożyć' nową wersję: set_registered_model_alias(..., alias='champion', version=2)
Serwis produkcyjny ładuje zawsze '@champion' — nie wie nic o numerach wersji
